# HelloClaw — персонализированный AI-ассистент

## Краткое описание проекта

HelloClaw — персонализированное AI-приложение на Hello-Agents.

**Основные возможности:**
- Поддержка настройки Agent личность и личность
- Автоматическое управление долговременной и суточной памятью
- Потоковый вызов инструмента, обратная связь в режиме реального времени о статусе выполнения
- Несколько сессий с историей

## Об авторе
- автор: tino-chen
- GitHub: [@tino-chen](https://github.com/tino-chen)
- дата: 2025-03

---
## Часть 1: настройка среды

In [1]:
# Установка зависимостей (при необходимости)
# !pip install -q hello-agents fastapi uvicorn python-dotenv pydantic httpx[socks]

In [2]:
import os
import sys
from dotenv import load_dotenv

# Добавить путь к проекту
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

# Загрузка переменных среды
load_dotenv()

# Конфигурация LLM (замените своим API ключ)
# Способ 1: переменные среды
# Способ 2: прямая установка
# os.environ["LLM_MODEL_ID"] = "glm-4"
# os.environ["LLM_API_KEY"] = "your-api-key"
# os.environ["LLM_BASE_URL"] = "https://open.bigmodel.cn/api/paas/v4/"

print("Настройка среды завершена!")

---
## Часть 2: импорт модулей и основных классов

In [3]:
from hello_agents import Config
from hello_agents.tools import ToolRegistry, ReadTool, WriteTool, CalculatorTool
from hello_agents.core.llm import HelloAgentsLLM

# импортировать HelloClaw основной модуль
from src.agent.helloclaw_agent import HelloClawAgent

print("Модули успешно импортированы!")

---
## Часть 3: определение пользовательских инструментов

HelloClaw: пользовательские инструменты.

In [4]:
# HelloClawAgent Инструкция по применению
# 
# HelloClawAgent автоматически:
# 1. Инициализировать рабочую область (~/.helloclaw/workspace）
# 2. Промпты из AGENTS.md, IDENTITY.md и др.
# 3. Регистрация инструментов
# 4. Система памяти
#
# Ключевые инструменты включают в себя:
# - Read/Write/Edit: файлы и MEMORY.md
# - python_calculator: математические расчеты
# - memory_*: Управление памятью (дневная память, поиск, список и т. д.)
# - exec_*: выполнение команды
# - search_web/fetch_url: Веб-поиск и сканирование

print("HelloClawAgent Описание инструмента загружено!")

---
## Часть 4: создание агента

использовать HelloAgents Фреймворк создает агента с возможностью вызова инструментов.

In [5]:
# создавать HelloClawAgent
# 
# HelloClawAgent автоматически:
# - Инициализировать рабочую область ~/.helloclaw/workspace
# - LLM из .env или config.json
# - Все инструменты
# - Системные промпты

agent = HelloClawAgent()

print(f"агент '{agent.name}' Создано успешно!")
print(f"Workspace: {agent.workspace_path}")
print(f"Доступные инструменты: {list(agent.tool_registry._tools.keys())[:10]}...")  # первые 10

---
## Часть 5: демонстрация функций

демонстрация HelloClaw основные функции.

In [6]:
# Пример 1: идентичность
# HelloClawAgent Поддерживает настройку идентификационной информации посредством разговоров и автоматическое сохранение ее в рабочей области.
print("="*50)
print("Пример 1: Бутстрап идентификации - настраивать Agent личность")
print("="*50)

# Шаг первый: спросите AI кто такой
print("【пользователь】: Кто ты?")
response = agent.chat("Кто ты?")
print(f"【Teddy】: {response}")

print("\n" + "-"*50 + "\n")

# Шаг второй: расскажите AI его идентичность
print("【пользователь】: твое имя Тедди, ты супер умный помощник, ты дружелюбный, профессиональный и отзывчивый.")
response = agent.chat("Тебя зовут Teddy, ты дружелюбный ассистент. Запомни.")
print(f"【Teddy】: {response}")

In [7]:
# Пример 2: вызов инструмента - калькулятор
print("="*50)
print("Пример 2: вызов инструмента - калькулятор")
print("="*50)

response = agent.chat("Посчитай (123 + 456) * 2")
print(f"\nОтветить: {response}")

In [8]:
# Пример 3: Управление памятью
print("="*50)
print("Пример 3: Управление памятью")
print("="*50)

# HelloClawAgent Имеется полноценная система управления памятью:
# - memory_add
# - memory_search: Поиск в памяти
# - memory_list: Список всех файлов памяти
# - Read/Write инструмент: Оперативная долговременная память MEMORY.md

# Добавьте ежедневное воспоминание
print("【Добавить ежедневную память】Использовать memory_add инструмент:")
print("-" * 40)
response = agent.chat("memory_add: пользователь сказал, что у него кошка Хуахуа")
print(f"результат: {response[:300]}..." if len(response) > 300 else f"результат: {response}")

print("\n" + "="*50)

# Список файлов памяти
print("[Список файлов памяти] используйте memory_list инструмент:")
print("-" * 40)
response = agent.chat("memory_list: список файлов")
print(f"результат: {response[:400]}..." if len(response) > 400 else f"результат: {response}")

---
## Часть 6: демонстрация потокового вывода

выставка HelloClaw Возможность вызова инструментов потоковой передачи.

In [9]:
import asyncio
from hello_agents.core.streaming import StreamEventType

async def demo_streaming():
    """Демонстрационный потоковый вывод - использовать HelloClawAgent из achat метод"""
    print("="*50)
    print("Демонстрация потокового вывода")
    print("="*50)
    
    # использовать HelloClawAgent из achat Методы потоковой передачи разговоров
    async for event in agent.achat("Посчитай 100 / 4 + 25"):
        if event.type == StreamEventType.LLM_CHUNK:
            chunk = event.data.get("chunk", "")
            print(chunk, end="", flush=True)
        
        elif event.type == StreamEventType.TOOL_CALL_START:
            tool_name = event.data.get("tool_name")
            print(f"\n[Инструмент вызова: {tool_name}]", flush=True)
        
        elif event.type == StreamEventType.TOOL_CALL_FINISH:
            result = event.data.get("result", "")
            preview = result[:100] + "..." if len(result) > 100 else result
            print(f"[Результаты инструмента: {preview}]", flush=True)
    
    print("\n" + "="*50)

# Запустите потоковую демонстрацию
await demo_streaming()

---
## Часть 7: итоги и перспективы

### Краткое описание проекта

**Реализованные функции:**
- на основе HelloAgents Платформа для умных разговоров
- Пользовательская система инструментов (выполнение команд, управление памятью и т. д.)
- Потоковая передача вызовов и вывода инструментов
- Управление сеансами и сохранение истории

**Возникшие проблемы и решения:**
1. **Вызов инструмента потоковой передачи** - в расширении HelloAgentsLLM реализует потоковый вызов инструментов
2. **управление памятью** - Спроектировал иерархическую систему памяти (долговременная память). + ежедневная память)
3. **Настройка личности** - использовать Markdown Файлы конфигурации обеспечивают гибкую настройку удостоверений.

### Направления будущих улучшений

- [ ] Поддержка мультимодального ввода (изображения, файлы)
- [ ] Добавьте больше встроенных инструментов
- [ ] поддерживать Agent сотрудничество среди
- [ ] Добавьте возможности голосового взаимодействия

---

**благодарный Datawhale сообщество и Hello-Agents проект!**